# ADB parser walkthrough

This notebook is the practical guide for parsing Asian Development Bank Excel workbooks in MARIO.

## What this notebook covers

- where to download the economic and air-emissions workbooks;
- when to pass a direct file path and when to pass a directory;
- how `year=` and `economies=` are used to disambiguate MRIO releases;
- how SRIO workbooks differ from MRIO workbooks;
- how `add_extensions=` imports the ADB air-emissions table;
- which parser warnings can appear and where to inspect them.

## Relevant source pages

- Official ADB MRIO and SRIO page: [ADB globalization portal](https://kidb.adb.org/globalization/current)
- Official ADB air-emissions extensions page: [ADB environmentally extended MRIOT](https://kidb.adb.org/globalization/adb_environmentally_extended_multiregional_inputoutput_tables)

MARIO does not download ADB workbooks automatically. The expected workflow is to download the `.xlsx` files manually from those pages and then point the parser to the local files.

## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_adb(...)`

ADB parsing currently supports only `IOT` tables, but the same public function works with:

- ADB `MRIO` workbooks;
- ADB `SRIO` workbooks;
- optional ADB air-emissions workbooks passed through `add_extensions=`.

## Key arguments

The key public arguments are:

- `path`: one ADB workbook or one directory containing multiple workbooks
- `table`: currently only `"IOT"` is supported
- `year`: used to disambiguate MRIO releases in one directory; mandatory for SRIO workbooks because one file contains multiple yearly sheets
- `economies`: used when a directory contains multiple MRIO variants for the same year, such as `62`, `72`, or `74` economies
- `add_extensions`: optional path to the ADB air-emissions workbook; this populates `E` and keeps `EY` zero-filled

## Direct file path vs directory path

Use a **direct workbook path** when you already know the exact file to parse. This is the simplest path and usually means you do not need `year=` or `economies=`.

Use a **directory path** when you keep several ADB workbooks together. In that case MARIO scans the directory, finds the candidate workbook files, and then uses `year=` and optionally `economies=` to pick the correct one.

## Expected local layout and caveats

Typical MRIO directory layout:

```text
ADB/
├── 62 economies/
│   └── ADB-MRIO-2024_*.xlsx
├── 72 economies/
│   └── ADB-MRIO72-2024_*.xlsx
└── CO2/
    └── 2023 EE-MRIOT (Air Emissions).xlsx
```

Typical SRIO workbook layout:

```text
ADB/SRIO/
└── CAN IOT 2000, 2007-2024.xlsx
```

When `path` is a directory, use `year=` and, when needed, `economies=` to disambiguate the workbook. For SRIO workbooks, `year=` is mandatory because one file contains multiple yearly sheets. When `add_extensions=` is used, MARIO warns if the emissions workbook year or regional coverage does not match the economic table.

In [1]:
import mario

## Parse one explicit MRIO workbook

Use this when you want to point MARIO to one specific ADB MRIO workbook on disk.

In [ ]:
db_mrio = mario.parse_adb(
    path="/path/to/ADB-MRIO-2024-August 2025.xlsx",
)

INFO Parser: reading ADB workbook ADB-MRIO-2024-August 2025.xlsx sheet ADB MRIO 2024.


INFO Parser: ADB MRIO parsed with 75 regions, 35 sectors, 375 final-demand columns and 6 factor rows.


INFO Metadata: initialized.


## Parse from a directory containing multiple MRIO releases

This is the important case when one directory contains several ADB MRIO files. Typical examples are:

- multiple years in the same folder;
- multiple release variants for the same year, such as `62`, `72`, or `74` economies.

In those cases, `year=` narrows the release year and `economies=` narrows the workbook variant.

In [3]:
db_mrio = mario.parse_adb(
    path="/path/to/ADB",
    year=2024,
    economies=74,
)

INFO Parser: reading ADB workbook ADB-MRIO-2024-August 2025.xlsx sheet ADB MRIO 2024.


INFO Parser: ADB MRIO parsed with 75 regions, 35 sectors, 375 final-demand columns and 6 factor rows.


INFO Metadata: initialized.


If the directory contains only one MRIO workbook for the requested year, `year=` may be enough on its own.

In [4]:
db_mrio = mario.parse_adb(
    path="/path/to/ADB",
    year=2023,
    economies=72,
)

INFO Parser: reading ADB workbook ADB-MRIO72-2023_July 2025.xlsx sheet ADB MRIO 2023.


INFO Parser: ADB MRIO parsed with 73 regions, 35 sectors, 365 final-demand columns and 6 factor rows.


INFO Metadata: initialized.


## Parse one SRIO workbook

ADB SRIO workbooks differ from MRIO workbooks in one key way: **one file contains multiple yearly sheets**. Because of that, `year=` is mandatory for SRIO parsing.

From the user side, the parser call stays simple: you still use `mario.parse_adb(...)`, but now `year=` selects the annual sheet inside the workbook.

In [5]:
db_srio = mario.parse_adb(
    path="/path/to/ITA IOT 2000, 2007-2024.xlsx",
    year=2024,
)

INFO Parser: reading ADB workbook ITA IOT 2000, 2007-2024.xlsx sheet 2024.


INFO Parser: ADB SRIO parsed with 35 sectors, 6 final-demand columns and 7 factor rows.


INFO Metadata: initialized.


## Add the ADB air-emissions extensions

When the matching environmental workbook is available locally, pass it through `add_extensions=`. MARIO imports the environmental extension matrix `E` from that workbook and keeps `EY` zero-filled.

The same argument works for both MRIO and SRIO economic tables.

In [2]:
db_mrio_ext = mario.parse_adb(
    path="/path/to/ADB-MRIO-2024-August 2025.xlsx",
    add_extensions="/path/to/2023 EE-MRIOT (Air Emissions).xlsx",
)

INFO Parser: reading ADB workbook ADB-MRIO72-2023_July 2025.xlsx sheet ADB MRIO 2023.
INFO Parser: ADB MRIO parsed with 73 regions, 35 sectors, 365 final-demand columns and 6 factor rows.
INFO Parser: importing ADB air-emissions extensions from 2023 EE-MRIOT (Air Emissions).xlsx sheet 2023.
INFO Metadata: initialized.


In [3]:
db_mrio_ext

name = ADB MRIO 2023 (72 economies)
table = IOT
scenarios = ['baseline']
Factor of production = 6
Satellite account = 80
Consumption category = 5
Region = 73
Sector = 35

In [7]:
db_srio_ext = mario.parse_adb(
    path="/path/to/ITA IOT 2000, 2007-2024.xlsx",
    year=2024,
    add_extensions="/path/to/2023 EE-MRIOT (Air Emissions).xlsx",
)

INFO Parser: reading ADB workbook ITA IOT 2000, 2007-2024.xlsx sheet 2024.


INFO Parser: ADB SRIO parsed with 35 sectors, 6 final-demand columns and 7 factor rows.


INFO Parser: importing ADB air-emissions extensions from 2023 EE-MRIOT (Air Emissions).xlsx sheet 2023.


WARNING Parser: ADB emissions year 2023 does not match the economic table year 2024.


INFO Metadata: initialized.


## Parser warnings you may see

When `add_extensions=` is used, MARIO can record parser warnings in the metadata history. The two relevant cases are:

- the emissions workbook year does not match the economic table year;
- the emissions workbook does not cover all regions present in the economic table.

The second case matters mainly for MRIO releases where the economic table contains regions not present in the emissions workbook. The parse still succeeds, but MARIO records the mismatch.

A simple way to inspect those parser notes is to check the metadata history after parsing:

In [8]:
db_mrio_ext.meta_history

[2026-04-29 10:50:11]    Table added into metadata with value equal to IOT.
[2026-04-29 10:50:11]    Price added into metadata with value equal to Current prices.
[2026-04-29 10:50:11]    Source added into metadata with value equal to Asian Development Bank MRIO Excel workbooks via https://kidb.adb.org/globalization/current.
[2026-04-29 10:50:11]    Year added into metadata with value equal to 2024.
[2026-04-29 10:50:11]    Tech_Assumption added into metadata with value equal to None.
[2026-04-29 10:50:11]    Parser: ADB emissions year 2023 does not match the economic table year 2024.
[2026-04-29 10:50:11]    Parser: ADB emissions workbook does not cover database regions ['TAJ', 'UZB']; zero satellite values were kept for those regions.


## Inspect the parsed database

Once parsed, the result is a standard MARIO database. At that point the normal exploration tools apply.

In [9]:
db_mrio_ext

name = ADB MRIO 2024 (74 economies)
table = IOT
scenarios = ['baseline']
Factor of production = 6
Satellite account = 80
Consumption category = 5
Region = 75
Sector = 35